# CamScanner model training (auto)
Seg + enhance + cls training and TFLite conversion. Synthetic data fallbacks - no dataset downloads.

In [ ]:
import os, subprocess, sys, traceback
print('PY', sys.version)
print('GPU:', os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip())
os.chdir('/kaggle/working')
if not os.path.isdir('camscanner/.git'):
    subprocess.run(['git','clone','-q','--depth','1','https://github.com/shubhambelbase/camscanner-train.git','camscanner'], check=True)
os.chdir('camscanner')
print('repo ready:', os.getcwd())

In [ ]:
import subprocess
r = subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless','Pillow','tqdm'], capture_output=True, text=True)
print('pip rc', r.returncode, r.stderr[-300:] if r.returncode else '')
import tensorflow as tf
print('tf', tf.__version__, 'gpus', tf.config.list_physical_devices('GPU'))

In [ ]:
import os
os.makedirs('/out', exist_ok=True)
os.makedirs('/data', exist_ok=True)
print('dirs ready')

In [ ]:
def run(name, *args):
    print(f'=== [{name}] START ===', flush=True)
    r = subprocess.run([sys.executable]+list(args), capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode:
        print(f'[{name}] FAILED rc={r.returncode}')
        print(r.stderr[-2000:])
    else:
        print(f'[{name}] OK', flush=True)
    return r.returncode

rc = run('seg', 'train/seg_train.py', '--data', '/data', '--out', '/out', '--epochs', '6')
rc = run('enhance', 'train/enhance_train.py', '--out', '/out') or rc
rc = run('cls', 'train/cls_train.py', '--data', '/data', '--out', '/out', '--epochs', '6') or rc
rc = run('tflite', 'train/convert_tflite.py', '--data', '/data', '--out', '/out') or rc
print('=== ALL_STEPS_DONE rc=%s ===' % rc, flush=True)

In [ ]:
import glob, shutil, os
os.chdir('/kaggle/working')
for f in glob.glob('/out/*'):
    if f.endswith('.tflite') or f.endswith('.txt') or f.endswith('.keras'):
        shutil.copy(f, '/kaggle/working/')
        print('copied', os.path.basename(f), os.path.getsize(f))
print('=== OUTPUT_COPY_DONE ===', flush=True)